<a href="https://colab.research.google.com/github/quantumguy-TR/QML-Practices/blob/main/QML_Hastane_Ilac_Etkinligi_Colab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applied QML: Hastane Senaryosu — İlaç Etkinliği Tahmini
## Kuantum Makine Öğrenmesi Uygulama Rehberi

**Eğitmen:** Joseph Volkan Özcan

Bu notebook, bir hastanede hasta verilerinin (kişisel bilgiler, tahlil sonuçları, MR/görüntüleme özellik vektörleri)  
kuantum bilgisayara güvenli şekilde gönderilerek **ilaç etkinliği tahmini** yapılmasını uçtan uca gösterir.

### Senaryo Özeti
1. Hasta verileri toplanır (demografik + laboratuvar + görüntüleme özellikleri)
2. **Anonimleştirme** (KVKK / GDPR uyumlu) uygulanır
3. Yüksek boyutlu veriler **boyut indirgeme (PCA)** ile küçültülür
4. Veri kuantum devresine **encoding** ile aktarılır
5. Kuantum sınıflandırıcı / kernel ile **ilaç etkinliği** (etkili / etkisiz veya etkinlik skoru) tahmin edilir
6. Sonuçlar ölçülür, yorumlanır ve (yetkili erişimde) de-anonimleştirilebilir

> Sadece **Python + IBM Qiskit** kullanılarak tasarlanmıştır. Colab’da çalışacak şekilde hazırlanmıştır.


## AŞAMA 1: Proje Tanımı, Veri Büyüklüğü ve Seçim Aracı

### 1.1 Problemin Tanımı
Bu projede çözülmek istenen problem:

> **İlaç Etkinliği Tahmini (Drug Efficacy Prediction)**  
> Belirli bir hasta profili (yaş, cinsiyet, tahlil değerleri, MR’dan türetilmiş özellikler vb.) verildiğinde,  
> seçilen ilacın **etkili olup olmayacağını** (sınıflandırma) veya **etkinlik skorunu** (regresyon) tahmin etmek.

Bu bir **Sınıflandırma** (veya isteğe bağlı Regresyon) problemidir. Optimizasyon (ör. doz ayarlama) ikinci aşamada düşünülebilir.

### 1.2 “Veri Büyüklüğü” Ne Anlama Gelir?

Kuantum makine öğrenmesinde “veri büyüklüğü” iki ayrı boyutta ele alınır. Bunları karıştırmamak kritiktir:

| Kavram | Anlamı (Hastane Senaryosunda) | Kuantum Etkisi |
|--------|-------------------------------|----------------|
| **n_samples (kayıt / hasta sayısı)** | Kaç adet hasta kaydı var? (ör. 200, 1000, 5000) | Kernel yöntemlerinde (QSVM) maliyet ≈ O(n²). Çok büyük n → simülatörde yavaşlar, gerçek cihazda pahalılaşır. |
| **n_features (özellik / nitelik sayısı)** | Her hasta için kaç sayısal özellik kullanılıyor? | **Doğrudan kübit sayısını belirler.** Angle Encoding’de 1 özellik ≈ 1 kübit. |
| **Ham veri boyutu** | MR görüntüsü (MB/GB), ham tahlil tabloları | Kuantuma **gönderilmez**. Önce klasik ön-işleme + özellik çıkarımı yapılır. |

#### Hastane verisinde tipik özellik kaynakları
- **Demografik / kişisel:** yaş, cinsiyet, BMI, komorbidite sayısı → düşük boyut
- **Laboratuvar tahlilleri:** hemogram, biyokimya, tümör belirteçleri → 10–50 özellik
- **Görüntüleme (MR / BT / PET):** ham piksel verisi **doğrudan kullanılmaz**.  
  Klasik bir CNN veya radiomics ile özellik vektörü çıkarılır (ör. 100–512 boyut).  
  Bu vektör daha sonra PCA ile 4–8 boyuta indirilir.

> **Özet kural (NISQ dönemi):**  
> - n_features > 8–10 ise **mutlaka PCA** uygulayın.  
> - n_samples çok büyükse (binlerce) önce alt-örnekleme veya mini-batch düşünün.  
> - Ham görüntü / kişisel kimlik bilgisi **asla** kuantum devresine ham haliyle gitmez.

### 1.3 Anonimleştirme (KVKK / GDPR)
Kuantuma gönderilmeden önce:
- Doğrudan tanımlayıcılar (TC, ad-soyad, protokol no) kaldırılır veya hash’lenir
- Kvası-tanımlayıcılar (nadir hastalık + yaş + posta kodu kombinasyonu) genelleştirilir
- Sadece model için gerekli sayısal özellikler ve anonim hasta ID’si tutulur

Sonuçlar yalnızca yetkili hekim / sistem tarafından de-anonimleştirilebilir (Aşama 10).

Aşağıdaki **interaktif seçim aracı** ile problem tipinizi, özellik sayısını, hasta (kayıt) sayısını ve veri tipini seçerek  
uygun **encoding**, **kübit ihtiyacı**, **algoritma** ve **cihaz** önerisini alabilirsiniz.


In [ ]:
# ============================================================
# AŞAMA 1 - İnteraktif Seçim Aracı
# (Encoding + Qubit + Algoritma + Cihaz) — İlaç Etkinliği
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import math

problem_type = widgets.Dropdown(
    options=[
        ("Sınıflandırma — İlaç etkili / etkisiz", "classification"),
        ("Regresyon — Etkinlik skoru (0-1 veya %)", "regression"),
        ("Optimizasyon — Doz / kombinasyon arama", "optimization")
    ],
    value="classification",
    description="Problem:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="520px")
)

n_features = widgets.IntSlider(
    value=12, min=2, max=50, step=1,
    description="Özellik (n_features):",
    style={"description_width": "140px"}, continuous_update=False
)

n_samples = widgets.IntSlider(
    value=300, min=50, max=5000, step=50,
    description="Hasta sayısı (n):",
    style={"description_width": "140px"}, continuous_update=False
)

data_type = widgets.Dropdown(
    options=[
        ("Sürekli (tahlil + radiomics özellikler)", "continuous"),
        ("Kategorik (cinsiyet, komorbidite, ilaç grubu)", "categorical"),
        ("Karışık (demografik + tahlil + görüntü özellikleri)", "mixed")
    ],
    value="mixed",
    description="Veri Tipi:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="520px")
)

priority = widgets.Dropdown(
    options=[
        ("Hızlı prototip / Eğitim (Simülatör)", "prototype"),
        ("Gerçek IBM Quantum denemesi", "hardware"),
        ("Yüksek doğruluk (simülatör + error mitigation)", "accuracy")
    ],
    value="prototype",
    description="Öncelik:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="520px")
)

run_button = widgets.Button(
    description="Encoding + Qubit + Algoritma & Cihaz Öner",
    button_style="success", icon="check",
    layout=widgets.Layout(width="400px", height="40px")
)
output_area = widgets.Output()

# Aşama 1b ile paylaşılacak ölçek
SELECTED_N_FEATURES = 12
SELECTED_N_SAMPLES = 300

def compute_encoding_and_qubits(features, samples, dtype):
    results = {
        "primary": None, "alternatives": [],
        "pca_recommended": features > 8,
        "pca_target": min(6, max(2, features // 2)) if features > 8 else features,
        "warnings": []
    }
    angle_qubits = features
    amp_qubits = math.ceil(math.log2(max(features, 2)))
    amp_padded = 2 ** amp_qubits

    if dtype == "continuous":
        if features <= 8:
            results["primary"] = (
                "Angle Encoding + ZZFeatureMap", angle_qubits,
                "Tahlil ve radiomics sürekli değerleri dönme açılarına yazılır. NISQ için stabil.",
                "zz_feature_map(feature_dimension=n, reps=2, entanglement='linear')"
            )
            results["alternatives"].append(("PauliFeatureMap", angle_qubits, "Daha zengin kernel için alternatif."))
            results["alternatives"].append((
                "Amplitude Encoding", amp_qubits,
                f"~{amp_qubits} kübit (2^{amp_qubits}={amp_padded} pad). State preparation pahalı."
            ))
        else:
            results["primary"] = (
                "PCA → Angle Encoding + ZZFeatureMap", results["pca_target"],
                f"Özellik ({features}) yüksek. PCA → {results['pca_target']} boyut, sonra Angle/ZZ.",
                "PCA(n_components=k) + zz_feature_map(feature_dimension=k, reps=2)"
            )
            results["alternatives"].append(("Amplitude Encoding", amp_qubits, f"Doğrudan {amp_qubits} kübit; derin hazırlama."))
            results["warnings"].append(f"n_features={features} > 8 → PCA uygulayın.")
    elif dtype == "categorical":
        results["primary"] = (
            "One-Hot (klasik) → Angle / Basis Encoding", angle_qubits,
            "Kategorikler One-Hot ile sayısallaştırılır, sonra Angle Encoding.",
            "pd.get_dummies(...) → zz_feature_map"
        )
        results["alternatives"].append(("Basis Encoding", "log2(kategori)", "Düşük kardinalitede daha az kübit."))
        results["warnings"].append("One-Hot kübit sayısını artırabilir.")
    else:
        results["primary"] = (
            "Hibrit: Kategorik→One-Hot + Sürekli→Angle/ZZ (+ PCA)", angle_qubits,
            "Hastane karışık veri için tipik yol. Yüksek boyutta PCA şart.",
            "One-Hot + StandardScaler + (PCA) + zz_feature_map"
        )
        if features > 6:
            results["pca_recommended"] = True
            results["warnings"].append("Karışık klinik veri → PCA ile 4–6 boyut önerilir.")

    if samples > 1000:
        results["warnings"].append(f"n_samples={samples} büyük → QSVM O(n²); alt-örnekleme düşünün.")
    if samples < 80:
        results["warnings"].append("Kohort küçük → overfitting riski; cross-validation kullanın.")
    return results

def get_recommendations(prob, features, samples, dtype, prio):
    recs = {
        "algorithms": [], "devices": [], "notes": [],
        "encoding_info": compute_encoding_and_qubits(features, samples, dtype),
        "pca_required": features > 8
    }
    enc = recs["encoding_info"]
    if prob == "classification":
        recs["algorithms"] = [
            ("FidelityQuantumKernel + QSVM", "İlaç etkili/etkisiz için ana yöntem."),
            ("VQC (RealAmplitudes)", "Eğitilebilir varyasyonel sınıflandırıcı."),
            ("SamplerQNN / EstimatorQNN", "Esnek kuantum sinir ağı.")
        ]
    elif prob == "regression":
        recs["algorithms"] = [
            ("Variational Quantum Regressor", "Etkinlik skoru tahmini."),
            ("Quantum Kernel Ridge", "Kernel regresyon."),
            ("EstimatorQNN regresyon", "Beklenti değeri çıktısı.")
        ]
    else:
        recs["algorithms"] = [
            ("QAOA", "Doz / kombinasyon arama."),
            ("VQE", "Maliyet minimizasyonu."),
            ("Grover", "Yapılandırılmış arama (ileri).")
        ]

    effective_q = enc["primary"][1]
    if isinstance(effective_q, str):
        effective_q = features

    if prio == "prototype":
        recs["devices"] = [
            ("AerSimulator / StatevectorSampler", "Geliştirme varsayılanı."),
            ("Qiskit Runtime Simulator", "Bulut simülasyonu.")
        ]
        recs["notes"].append("Önce simülatör, sonra gerçek donanım.")
    elif prio == "hardware":
        if isinstance(effective_q, int) and effective_q <= 5:
            recs["devices"] = [("IBM Quantum küçük backend", "≤5 kübit."), ("AerSimulator ön test", "Önce simülatör.")]
        elif isinstance(effective_q, int) and effective_q <= 10:
            recs["devices"] = [("IBM 127+ kübit + error mitigation", "PCA sonrası."), ("Runtime SamplerV2/EstimatorV2", "Modern primitives.")]
            recs["notes"].append("reps=1–2, ResilienceLevel kullanın.")
        else:
            recs["devices"] = [("PCA → 4–6 özellik sonra donanım", "12+ pratik değil."), ("Aer + noise model", "Gürültü taklidi.")]
    else:
        recs["devices"] = [
            ("StatevectorSampler (exact)", "Gürültüsüz referans."),
            ("Aer + yüksek shot", "İstatistiksel kararlılık."),
            ("Donanım + ZNE/PEC", "Son aşama.")
        ]

    recs["notes"].append("Kişisel tanımlayıcılar kuantum devresine gönderilmez (KVKK).")
    return recs

def on_button_clicked(b):
    global SELECTED_N_FEATURES, SELECTED_N_SAMPLES
    with output_area:
        clear_output()
        SELECTED_N_FEATURES = n_features.value
        SELECTED_N_SAMPLES = n_samples.value
        recs = get_recommendations(
            problem_type.value, n_features.value, n_samples.value,
            data_type.value, priority.value
        )
        enc = recs["encoding_info"]
        display(HTML("<h3 style='color:#1a73e8;'>🔍 Öneri Sonuçları — İlaç Etkinliği</h3>"))
        print(f"Problem: {problem_type.label}")
        print(f"n_features={n_features.value} | n_samples={n_samples.value} | {data_type.label}")
        print("=" * 70)
        name, qubits, desc, hint = enc["primary"]
        display(HTML("<b>📐 Encoding & Qubit</b>"))
        print(f"▶ {name}\n   Kübit: {qubits}\n   {desc}\n   Qiskit: {hint}\n")
        if enc["alternatives"]:
            print("Alternatifler:")
            for a, q, d in enc["alternatives"]:
                print(f"  • {a} | kübit≈{q} — {d}")
        f = n_features.value
        print(f"\nKarşılaştırma: Angle={f} | Amplitude={math.ceil(math.log2(max(f,2)))} kübit")
        if enc["pca_recommended"]:
            display(HTML(f"<div style='background:#fff3cd;padding:8px;border-radius:6px;'>"
                         f"⚠️ PCA önerilir → hedef ~{enc['pca_target']} özellik</div>"))
        for w in enc["warnings"]:
            print(f"  ⚠ {w}")
        print()
        display(HTML("<b>🧠 Algoritmalar</b>"))
        for i, (n, d) in enumerate(recs["algorithms"], 1):
            print(f"  {i}. {n} — {d}")
        display(HTML("<b>💻 Cihazlar</b>"))
        for i, (n, d) in enumerate(recs["devices"], 1):
            print(f"  {i}. {n} — {d}")
        display(HTML("<b>📝 Notlar</b>"))
        for note in recs["notes"]:
            print(f"  • {note}")
        display(HTML("<p><b>➡️ Sonraki:</b> Aşama 1b (CPU/GPU/TPU), sonra Aşama 2 kurulum.</p>"))

run_button.on_click(on_button_clicked)
display(widgets.VBox([
    widgets.HTML("<h3>🎯 İlaç Etkinliği — Problem & Encoding Seçim Aracı</h3>"),
    widgets.HTML("<p style='color:#555;'><b>n_features</b> = özellik/kübit adayı; <b>n_samples</b> = hasta sayısı. Ham MR boyutu burada seçilmez.</p>"),
    problem_type, n_features, n_samples, data_type, priority,
    widgets.HTML("<br>"), run_button, output_area
]))


## AŞAMA 1b: Simülasyon Ortamı Seçimi (CPU / GPU / TPU)

Colab menüsü (**Runtime → Change runtime type**) ile bu hücredeki seçim birlikte kullanılmalıdır.

| Seçim | Qiskit Aer | Bu ilaç etkinliği problemi (genelde 4–8 kübit) |
|-------|------------|-----------------------------------------------|
| **CPU** | Varsayılan | **Önerilen** — stabil, ek kurulum yok |
| **GPU** | `qiskit-aer-gpu` + `device="GPU"` | Büyük n_samples veya yüksek kübit denemelerinde denenebilir |
| **TPU / v5e-1** | Aer TPU **kullanmaz** | **Önerilmez** — TF/JAX içindir |

Sadece runtime’dan GPU/TPU seçmek Qiskit kodunu otomatik hızlandırmaz; aşağıdaki araç kurulum ve kod ipuçlarını üretir.


In [ ]:
# ============================================================
# AŞAMA 1b - CPU / GPU / TPU seçimi (probleme duyarlı)
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import subprocess, os

COMPUTE_BACKEND = "cpu"
USE_GPU_AER = False
AER_DEVICE = "CPU"

backend_dd = widgets.Dropdown(
    options=[
        ("CPU — önerilen (küçük/orta QML)", "cpu"),
        ("GPU — Aer GPU paketi ile", "gpu"),
        ("TPU / v5e-1 — Qiskit Aer için önerilmez", "tpu"),
    ],
    value="cpu",
    description="Ortam:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="520px"),
)

_default_f = int(globals().get("SELECTED_N_FEATURES", 6))
_default_n = int(globals().get("SELECTED_N_SAMPLES", 300))

feat_slider = widgets.IntSlider(
    value=min(max(_default_f, 2), 30), min=2, max=30,
    description="n_features≈", style={"description_width": "100px"}, continuous_update=False
)
samp_slider = widgets.IntSlider(
    value=min(max(_default_n, 50), 5000), min=50, max=5000, step=50,
    description="n_samples≈", style={"description_width": "100px"}, continuous_update=False
)

btn = widgets.Button(
    description="Ortamı Değerlendir ve Ayarla",
    button_style="primary", icon="cogs",
    layout=widgets.Layout(width="320px", height="40px"),
)
out = widgets.Output()

def _recommend(backend, n_feat, n_samp):
    notes, install_cmds = [], []
    code_hint, verdict = "", ""
    heavy = n_samp >= 800
    many_q = n_feat >= 12
    tiny = n_feat <= 8 and n_samp <= 500

    if backend == "cpu":
        verdict = "CPU — bu problem ölçeği için genelde en iyi denge."
        notes.append("StatevectorSampler / Aer CPU ek paketsiz çalışır.")
        if heavy:
            notes.append(f"n_samples≈{n_samp} yüksek → kernel O(n²) pahalı; alt-örnekleme veya GPU deneyin.")
        if many_q:
            notes.append(f"n_features≈{n_feat} yüksek → PCA ile 4–6 boyuta indirin.")
        install_cmds = ["!pip install -q qiskit qiskit-aer qiskit-machine-learning qiskit-algorithms"]
        code_hint = (
            "from qiskit.primitives import StatevectorSampler\n"
            "sampler = StatevectorSampler()\n"
            "# veya AerSimulator(method='statevector', device='CPU')"
        )
    elif backend == "gpu":
        verdict = "GPU — Colab runtime da GPU olmalı (Runtime → Change runtime type)."
        install_cmds = [
            "!pip install -q qiskit qiskit-machine-learning qiskit-algorithms",
            "!pip install -q qiskit-aer-gpu-cu11   # CUDA 11 Colab",
            "# alternatif CUDA 12: !pip install -q qiskit-aer-gpu",
        ]
        code_hint = (
            "from qiskit_aer import AerSimulator\n"
            "sim = AerSimulator(method='statevector', device='GPU')\n"
            "print(AerSimulator().available_devices())"
        )
        if tiny:
            notes.append("Küçük devrede GPU bazen CPU’dan yavaş kalır (transfer maliyeti).")
        if heavy or many_q:
            notes.append("Büyük kernel / yüksek özellik → GPU kazanç olasılığı artar.")
        notes.append("`!nvidia-smi` ile GPU’yu doğrulayın; gerekirse runtime Restart.")
        notes.append("StatevectorSampler varsayılanı CPU’dur; GPU için Aer device='GPU' gerekir.")
    else:
        verdict = "TPU — Qiskit Aer TPU kullanmaz; simülasyon CPU’da kalır."
        notes.append("TPU/v5e-1 TF-JAX içindir; bu QML yığınını hızlandırmaz.")
        notes.append("Öneri: Runtime’ı CPU veya (Aer GPU deneyecekseniz) GPU yapın.")
        install_cmds = ["!pip install -q qiskit qiskit-aer qiskit-machine-learning qiskit-algorithms"]
        code_hint = "from qiskit.primitives import StatevectorSampler\nsampler = StatevectorSampler()"

    return verdict, notes, install_cmds, code_hint

def on_click(_):
    global COMPUTE_BACKEND, USE_GPU_AER, AER_DEVICE
    with out:
        clear_output()
        COMPUTE_BACKEND = backend_dd.value
        USE_GPU_AER = COMPUTE_BACKEND == "gpu"
        AER_DEVICE = "GPU" if USE_GPU_AER else "CPU"
        verdict, notes, install_cmds, code_hint = _recommend(
            COMPUTE_BACKEND, feat_slider.value, samp_slider.value
        )
        display(HTML("<h3 style='color:#1a73e8;'>🖥️ Simülasyon Ortamı Sonucu</h3>"))
        print(f"Seçim: {backend_dd.label}")
        print(f"COMPUTE_BACKEND={COMPUTE_BACKEND} | USE_GPU_AER={USE_GPU_AER} | AER_DEVICE={AER_DEVICE}")
        print(f"Ölçek: n_features≈{feat_slider.value}, n_samples≈{samp_slider.value}")
        print("=" * 70)
        print(verdict)
        display(HTML("<b>📝 Notlar</b>"))
        for n in notes:
            print(f"  • {n}")
        display(HTML("<b>📦 Kurulum (Aşama 2 veya burada)</b>"))
        for c in install_cmds:
            print(c)
        display(HTML("<b>💻 Kod ipucu</b>"))
        print(code_hint)
        display(HTML("<b>🔎 Ortam kontrolü</b>"))
        try:
            print(subprocess.check_output(["nvidia-smi", "-L"], stderr=subprocess.STDOUT, text=True).strip())
        except Exception:
            print("nvidia-smi yok / GPU görünmüyor (CPU veya TPU runtime olabilir).")
        if os.environ.get("COLAB_TPU_ADDR"):
            print("COLAB_TPU_ADDR tanımlı → TPU runtime (Aer yine CPU kullanır).")
        display(HTML(
            "<div style='background:#e8f0fe;padding:10px;border-radius:6px;margin-top:8px;'>"
            "<b>Sonraki:</b> Aşama 2 kurulum. GPU seçtiyseniz aer-gpu paketini de kurun. "
            "Aşama 8 sampler <code>AER_DEVICE</code> bilgisini okur."
            "</div>"
        ))

btn.on_click(on_click)
display(widgets.VBox([
    widgets.HTML("<h3>⚙️ CPU / GPU / TPU Seçimi</h3>"),
    widgets.HTML("<p style='color:#555;'>Aşama 1’deki n_features / n_samples değerleri varsa otomatik dolar; değilse elle ayarlayın.</p>"),
    backend_dd, feat_slider, samp_slider, widgets.HTML("<br>"), btn, out
]))


## AŞAMA 2: Kütüphane Kurulumları ve Çevresel Ayarlar

- **CPU** (varsayılan / Aşama 1b): standart `qiskit-aer` yeterli.
- **GPU** (Aşama 1b): `qiskit-aer-gpu-cu11` (veya CUDA 12 için `qiskit-aer-gpu`) kurun; Colab runtime GPU olmalı.
- **TPU**: ek Qiskit paketi yok; Aer CPU kullanır.

IBM Quantum için API anahtarı `.env` veya Colab Secrets ile verilebilir.


In [ ]:
import os
import sys

_backend = globals().get("COMPUTE_BACKEND", "cpu")
_use_gpu = bool(globals().get("USE_GPU_AER", False))

print(f"Kurulum modu: COMPUTE_BACKEND={_backend}, USE_GPU_AER={_use_gpu}")

# Temel paketler (her zaman)
!pip install -q qiskit qiskit-machine-learning qiskit-ibm-runtime qiskit-algorithms
!pip install -q scikit-learn numpy pandas python-dotenv matplotlib seaborn

# Aer: GPU veya CPU
if _use_gpu:
    print("GPU Aer paketi kuruluyor (qiskit-aer-gpu-cu11)...")
    !pip install -q qiskit-aer-gpu-cu11
    print("Sorun olursa Runtime Restart sonrası tekrar deneyin; CUDA 12 için: qiskit-aer-gpu")
else:
    print("CPU Aer kuruluyor...")
    !pip install -q qiskit-aer

import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
ibm_api_key = os.getenv("IBM_QUANTUM_TOKEN")

print("Kütüphaneler yüklendi.")
print(f"IBM API anahtarı {'tanımlı' if ibm_api_key else 'tanımlı değil (simülatör)'}.")


Kurulum modu: COMPUTE_BACKEND=gpu, USE_GPU_AER=True
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.7/120.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.2/224.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 8.7 MB/s eta 0:00:00

## AŞAMA 3: Veri Tipi, Nitelikler ve Boyut İndirgeme (PCA)

### Hastane verisinde tipik nitelikler
- **Demografik (kategorik / sürekli):** yaş, cinsiyet, BMI, sigara, komorbidite sayısı  
- **Laboratuvar (sürekli):** WBC, CRP, kreatinin, tümör belirteçleri, karaciğer enzimleri…  
- **Görüntüleme türevi (sürekli, yüksek boyut):** Radiomics veya CNN embedding (50–512 boyut)

Ham MR/BT **doğrudan** kuantum devresine verilmez. Klasik özellik çıkarımı sonrası  
yüksek boyutlu vektör **PCA** ile 4–8 boyuta indirilir (NISQ kübit sınırı).

Aşağıdaki fonksiyon, standardizasyon + PCA uygular.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def boyut_indirgeme_uygula(X_veri, n_components=4):
    """
    Kuantum devresine uygun boyut için StandardScaler + PCA.
    X_veri: (n_samples, n_features) numpy array veya DataFrame
    """
    pipeline = make_pipeline(
        StandardScaler(),
        PCA(n_components=n_components)
    )
    X_kucultulmus = pipeline.fit_transform(X_veri)
    print(f"Orijinal boyut : {X_veri.shape}")
    print(f"İndirgenmiş   : {X_kucultulmus.shape}")
    print(f"Açıklanan varyans oranı (kümülatif): {pipeline.named_steps['pca'].explained_variance_ratio_.sum():.3f}")
    return X_kucultulmus, pipeline

# Örnek (rastgele yüksek boyutlu özellik → 4 boyuta)
np.random.seed(42)
X_ornek = np.random.randn(100, 30)  # 100 hasta, 30 radiomics/tahlil özelliği
X_4d, pca_pipe = boyut_indirgeme_uygula(X_ornek, n_components=4)


Orijinal boyut : (100, 30)
İndirgenmiş   : (100, 4)
Açıklanan varyans oranı (kümülatif): 0.262


### Kategorik alanların hazırlanması
Cinsiyet, ilaç grubu, komorbidite kodu gibi alanlar One-Hot Encoding ile sayısallaştırılmalıdır.


In [ ]:
# Kategorik → One-Hot örneği (hastane bağlamı)
ornek_df = pd.DataFrame({
    "Yas": [45, 62, 38],
    "Cinsiyet": ["K", "E", "K"],
    "Komorbidite": ["Diyabet", "Yok", "Hipertansiyon"]
})
df_one_hot = pd.get_dummies(ornek_df, columns=["Cinsiyet", "Komorbidite"], dtype=float)
print("One-Hot sonrası:")
print(df_one_hot)


One-Hot sonrası:
   Yas  Cinsiyet_E  Cinsiyet_K  Komorbidite_Diyabet  \
0   45         0.0         1.0                  1.0   
1   62         1.0         0.0                  0.0   
2   38         0.0         1.0                  0.0   

   Komorbidite_Hipertansiyon  Komorbidite_Yok  
0                        0.0              0.0  
1                        0.0              1.0  
2                        1.0              0.0  


## AŞAMA 4: Sentetik Hasta Verisi Üretimi ve Anonimleştirme (KVKK)

Gerçek hasta verisi paylaşmadan modeli geliştirebilmek için **sentetik kohort** üretiriz.  
Ardından KVKK/GDPR uyumlu anonimleştirme adımları uygulanır:

1. Doğrudan tanımlayıcılar (ad, TC, protokol) → hash veya tamamen kaldırılır  
2. Sadece model özellikleri + anonim ID tutulur  
3. Kuantum devresine **yalnızca anonim sayısal vektör** gider  

Aşağıda iki seçenek vardır:
- **Sentetik veri üret** (önerilen başlangıç)
- Kendi CSV’nizi yükleyin (Aşama 5)


In [ ]:
import hashlib
import uuid
from sklearn.datasets import make_classification

def sentetik_hasta_verisi_uret(n_samples=300, n_features=12, random_state=42):
    """
    İlaç etkinliği sınıflandırması için sentetik hasta kohortu.
    Özellikler: tahlil + radiomics benzeri sürekli değerler.
    Hedef: 0 = ilaç etkisiz, 1 = ilaç etkili
    """
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=max(2, n_features // 2),
        n_redundant=max(0, n_features // 4),
        n_clusters_per_class=2,
        class_sep=1.2,
        random_state=random_state
    )
    # Klinik isimlendirme (örnek)
    feature_names = [f"Ozellik_{i+1}" for i in range(n_features)]
    # İlk birkaçına anlamlı isim verelim
    klinik_isimler = [
        "Yas_norm", "BMI_norm", "CRP", "WBC", "Kreatinin",
        "Tumor_marker", "Radiomics_1", "Radiomics_2",
        "Radiomics_3", "Radiomics_4", "Karaciger_ALT", "Hgb"
    ]
    for i, isim in enumerate(klinik_isimler):
        if i < n_features:
            feature_names[i] = isim

    df = pd.DataFrame(X, columns=feature_names)
    df["Ilac_Etkin"] = y  # 0/1

    # Sahte doğrudan tanımlayıcılar (anonimleştirme demosu için)
    df["Hasta_Ad"] = [f"Hasta_{i}" for i in range(n_samples)]
    df["Protokol_No"] = [f"PR-{10000+i}" for i in range(n_samples)]
    df["TC_Kimlik"] = [f"{10000000000+i}" for i in range(n_samples)]

    return df

def anonimlestir(df, id_col="Anonim_ID"):
    """
    Doğrudan tanımlayıcıları kaldırır / hash’ler.
    Geri dönüşüm için sadece yetkili tarafta tutulacak mapping üretir.
    """
    mapping = {}
    anon_ids = []
    for idx, row in df.iterrows():
        ham = f"{row.get('TC_Kimlik','')}-{row.get('Protokol_No','')}-{row.get('Hasta_Ad','')}"
        anon = hashlib.sha256(ham.encode()).hexdigest()[:16]
        anon_ids.append(anon)
        mapping[anon] = {
            "Hasta_Ad": row.get("Hasta_Ad"),
            "Protokol_No": row.get("Protokol_No"),
            "TC_Kimlik": row.get("TC_Kimlik")
        }

    # Tanımlayıcı sütunları düşür
    drop_cols = [c for c in ["Hasta_Ad", "Protokol_No", "TC_Kimlik"] if c in df.columns]
    df_anon = df.drop(columns=drop_cols).copy()
    df_anon.insert(0, id_col, anon_ids)

    print(f"Anonimleştirme tamamlandı. {len(df_anon)} kayıt.")
    print(f"Kaldırılan sütunlar: {drop_cols}")
    print(f"Mapping sözlüğü {len(mapping)} girdi içeriyor (sadece yetkili tarafta saklanmalı).")
    return df_anon, mapping

# --- Sentetik kohort üret ve anonimleştir ---
df_ham = sentetik_hasta_verisi_uret(n_samples=300, n_features=12)
print("Ham sentetik veri (ilk 3 satır, tanımlayıcılar görünür):")
print(df_ham[["Hasta_Ad", "Protokol_No", "Yas_norm", "CRP", "Ilac_Etkin"]].head(3))
print()

df_anon, deanon_mapping = anonimlestir(df_ham)
print("\nAnonim veri (ilk 3 satır):")
print(df_anon.head(3))


Ham sentetik veri (ilk 3 satır, tanımlayıcılar görünür):
  Hasta_Ad Protokol_No  Yas_norm       CRP  Ilac_Etkin
0  Hasta_0    PR-10000  0.327880 -3.429009           0
1  Hasta_1    PR-10001  1.096469 -5.153065           0
2  Hasta_2    PR-10002 -1.643189  5.067190           0

Anonimleştirme tamamlandı. 300 kayıt.
Kaldırılan sütunlar: ['Hasta_Ad', 'Protokol_No', 'TC_Kimlik']
Mapping sözlüğü 300 girdi içeriyor (sadece yetkili tarafta saklanmalı).

Anonim veri (ilk 3 satır):
          Anonim_ID  Yas_norm  BMI_norm       CRP       WBC  Kreatinin  \
0  ba65e1d607189a7d  0.327880 -0.125454 -3.429009 -0.966273   0.622677   
1  693471c5ba2d6aa9  1.096469  1.223569 -5.153065  0.123042  -2.753199   
2  baf585ace757c3cc -1.643189 -0.103255  5.067190  0.676039   4.462769   

   Tumor_marker  Radiomics_1  Radiomics_2  Radiomics_3  Radiomics_4  \
0      1.057780    -0.633576    -2.640361    -1.195982     1.127387   
1     -2.309980     0.784641    -1.693481    -2.583432     0.700140   
2      1.021

## AŞAMA 5: (Opsiyonel) Kendi CSV Dosyanızı Yükleme

Sentetik veri yerine gerçek (anonimleştirilmiş) hasta CSV’si kullanmak isterseniz  
aşağıdaki yükleyiciyi kullanın. CSV’de hedef sütun adı tercihen `Ilac_Etkin` olsun.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import io

uploader = widgets.FileUpload(accept=".csv", multiple=False, description="CSV Yükle")
display(uploader)

def yuklenen_csv_oku(uploader_widget):
    if not uploader_widget.value:
        print("Henüz dosya yüklenmedi. Sentetik veri (df_anon) kullanılmaya devam eder.")
        return None
    # Colab / Jupyter farkı için esnek okuma
    uploaded = uploader_widget.value
    if isinstance(uploaded, dict):
        file_info = list(uploaded.values())[0]
        content = file_info["content"]
    else:
        file_info = uploaded[0]
        content = file_info["content"]
    df = pd.read_csv(io.BytesIO(content))
    print(f"Yüklendi: {df.shape[0]} satır, {df.shape[1]} sütun")
    print(df.head(3))
    return df

# Kullanım: df_yuklenen = yuklenen_csv_oku(uploader)


FileUpload(value={}, accept='.csv', description='CSV Yükle')

## AŞAMA 5b: Model İçin Özellik Matrisinin Hazırlanması

Anonim veri üzerinden:
1. Hedef sütunu ayır  
2. (Gerekirse) PCA uygula  
3. Eğitim / test bölünmesi yap


In [ ]:
from sklearn.model_selection import train_test_split

# Hedef ve özellikler
TARGET = "Ilac_Etkin"
feature_cols = [c for c in df_anon.columns if c not in (TARGET, "Anonim_ID")]

X = df_anon[feature_cols].values
y = df_anon[TARGET].values

print(f"Özellik matrisi: {X.shape}  |  Hedef dağılımı: {np.bincount(y)}")

# Boyut yüksekse PCA (seçim aracındaki öneriye göre ayarlayın)
N_COMPONENTS = 4  # Seçim aracından gelen pca_target değerini buraya yazabilirsiniz
if X.shape[1] > N_COMPONENTS:
    X_red, pca_model = boyut_indirgeme_uygula(X, n_components=N_COMPONENTS)
else:
    X_red = StandardScaler().fit_transform(X)
    print("PCA gerekmedi, sadece standardizasyon uygulandı.")

X_train, X_test, y_train, y_test = train_test_split(
    X_red, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Eğitim: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")


Özellik matrisi: (300, 12)  |  Hedef dağılımı: [151 149]
Orijinal boyut : (300, 12)
İndirgenmiş   : (300, 4)
Açıklanan varyans oranı (kümülatif): 0.651
Eğitim: 225  |  Test: 75


## AŞAMA 6–7: Feature Map ve Ansatz (Kuantum Devresi)

- **Feature Map:** Klasik özellikleri kuantum durumuna kodlar (ZZFeatureMap)  
- **Ansatz:** Eğitilebilir parametreli devre (RealAmplitudes) — VQC kullanıldığında  

Kernel tabanlı QSVM için yalnızca feature map yeterlidir.


In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import zz_feature_map, real_amplitudes

num_features = X_train.shape[1]
print(f"Kullanılacak kübit / özellik sayısı: {num_features}")

feature_map = zz_feature_map(
    feature_dimension=num_features,
    reps=2,
    entanglement="linear"
)

ansatz = real_amplitudes(num_qubits=num_features, reps=3)

# Birleşik devre örneği (VQC için)
qc = QuantumCircuit(num_features)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)
qc.measure_all()

print("Feature map + Ansatz hazır.")
print(f"Feature map derinliği (approx): {feature_map.depth()}")


Kullanılacak kübit / özellik sayısı: 4
Feature map + Ansatz hazır.
Feature map derinliği (approx): 19


## AŞAMA 8: Kuantum Kernel Oluşturma

FidelityQuantumKernel, veri noktalarının kuantum durumlarındaki benzerliğini hesaplar.  
`shots` değeri istatistiksel kararlılığı etkiler (1024 iyi bir başlangıçtır).


In [ ]:
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit.primitives import StatevectorSampler

shots = 1024
_device = globals().get("AER_DEVICE", "CPU")
_use_gpu = globals().get("USE_GPU_AER", False)

if _use_gpu:
    try:
        from qiskit_aer import AerSimulator
        sim = AerSimulator(method="statevector", device="GPU")
        print("Aer GPU denemesi OK. available_devices:", AerSimulator().available_devices())
    except Exception as e:
        print("GPU Aer kullanılamadı:", e)
    # Fidelity kernel için exact sampler (küçük devrede stabil)
    sampler = StatevectorSampler()
    print("FidelityQuantumKernel: StatevectorSampler (exact). Büyük devrede AerSampler+GPU özelleştirilebilir.")
else:
    sampler = StatevectorSampler()
    print(f"Sampler: StatevectorSampler (AER_DEVICE={_device})")

fidelity = ComputeUncompute(sampler=sampler)
qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)
print(f"Kuantum Kernel hazır. Shot referans: {shots}")


ModuleNotFoundError: No module named 'qiskit_machine_learning'

## AŞAMA 9: Modelin Eğitilmesi (QSVM) — Simülatör

Qiskit Machine Learning içindeki QSVC (Quantum Support Vector Classifier) kullanılır.  
Önce simülatörde eğitip test skoruna bakıyoruz.


In [ ]:
from qiskit_machine_learning.algorithms import QSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# QSVC — quantum kernel ile SVM
qsvc = QSVC(quantum_kernel=qkernel)

print("Eğitim başlıyor (simülatör)...")
qsvc.fit(X_train, y_train)

y_pred = qsvc.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\nTest doğruluğu: {acc:.3f}")
print("\nSınıflandırma raporu:")
print(classification_report(y_test, y_pred, target_names=["Etkisiz", "Etkili"]))
print("Karmaşıklık matrisi:")
print(confusion_matrix(y_test, y_pred))


ModuleNotFoundError: No module named 'qiskit_machine_learning'

## AŞAMA 10: Sonuçların Yorumlanması ve De-Anonimleştirme

Kuantum / model çıktısı anonim ID’ler üzerinden gelir.  
Sadece yetkili sistem, saklanan mapping ile gerçek kimliğe dönebilir.

> Üretim ortamında mapping **ayrı, erişim kontrollü** bir serviste tutulmalıdır.  
> Bu notebook yalnızca eğitim amaçlı demo yapar.


In [ ]:
# Örnek: test setinden birkaç anonim ID için tahmin + de-anonimleştirme
# (Gerçek pipeline'da X_test satırları ile Anonim_ID eşleştirilir)

# Demo: rastgele 3 anonim ID al
ornek_anon_ids = df_anon["Anonim_ID"].sample(3, random_state=1).tolist()

print("Örnek de-anonimleştirme (sadece yetkili erişim):")
for aid in ornek_anon_ids:
    gercek = deanon_mapping.get(aid, {})
    print(f"  Anonim ID : {aid}")
    print(f"  → Gerçek  : {gercek.get('Hasta_Ad')} | Protokol: {gercek.get('Protokol_No')}")
    print()


## AŞAMA 11: Sonuçların Dışa Aktarılması (CSV & SQL)


In [ ]:
import sqlite3
import os

# Tahmin sonuçlarını tabloya dök (demo: test seti)
sonuc_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred
})
sonuc_df["Dogru_Mu"] = (sonuc_df["y_true"] == sonuc_df["y_pred"]).astype(int)

output_dir = "./sonuclar"
os.makedirs(output_dir, exist_ok=True)
csv_path = os.path.join(output_dir, "ilac_etkinligi_tahminleri.csv")
sonuc_df.to_csv(csv_path, index=False)
print(f"CSV kaydedildi: {csv_path}")

conn = sqlite3.connect("qml_hastane.db")
sonuc_df.to_sql("Ilac_Etkinlik_Tahminleri", conn, if_exists="replace", index=False)
conn.close()
print("SQL tablosuna yazıldı: Ilac_Etkinlik_Tahminleri")
print(sonuc_df.head())


## AŞAMA 12: Görselleştirme

Sınıf dağılımı ve (varsa) kuantum ölçüm histogramı.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram

# 1) Gerçek vs tahmin sınıf dağılımı
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(y_test, bins=[-0.5, 0.5, 1.5], rwidth=0.6, color="steelblue")
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["Etkisiz", "Etkili"])
axes[0].set_title("Gerçek (y_test)")
axes[1].hist(y_pred, bins=[-0.5, 0.5, 1.5], rwidth=0.6, color="teal")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["Etkisiz", "Etkili"])
axes[1].set_title("Tahmin (y_pred)")
plt.tight_layout()
plt.show()

# 2) Örnek kuantum ölçüm histogramı (demo counts)
demo_counts = {"00": 480, "01": 120, "10": 95, "11": 329}
fig2 = plot_histogram(demo_counts, title="Örnek Kuantum Ölçüm Histogramı", figsize=(8, 5), color="teal")
display(fig2)


## Özet ve Sonraki Adımlar

Bu notebook’ta yapılanlar:

1. **Problem tanımı** — Hastane ilaç etkinliği tahmini (sınıflandırma)
2. **Aşama 1b** — CPU / GPU / TPU seçimi ve Qiskit Aer’e etkisi  
2. **Veri büyüklüğü ayrımı** — `n_samples` (hasta sayısı) vs `n_features` (özellik / kübit) netleştirildi  
3. **Seçim aracı** — Encoding, kübit hesabı, algoritma ve cihaz önerisi  
4. **Sentetik kohort** + **KVKK anonimleştirme**  
5. **PCA** ile boyut indirgeme  
6. **ZZFeatureMap + FidelityQuantumKernel + QSVC** ile eğitim ve test  
7. **De-anonimleştirme demosu**, CSV/SQL export, görselleştirme  

### Gerçek ortama geçerken
- Mapping tablosunu kuantum pipeline’ından **ayırın** (ayrı yetkili servis)  
- Ham görüntüleri yalnızca klasik özellik çıkarımı için kullanın  
- Gerçek IBM Quantum backend’ine geçmeden önce noise model ile test edin  
- Klinik validasyon ve etik kurul onayı olmadan üretim kararı vermeyin  

**Colab’da çalıştırma sırası:** Yukarıdan aşağıya hücreleri sırayla Run edin.  
Seçim aracındaki `n_features` / PCA hedefini Aşama 5b’deki `N_COMPONENTS` ile uyumlu tutun.
